# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreeyeshbaral/ShreeyeshAssignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane 4 — CTR / Engagement Opportunity Scoring → Ranking / Scoring task.**

The core question is *"which pages should be reviewed first?"* — and that maps to **ranking**. The output is an ordered queue of pages, ranked by how severely they under-capture clicks (or engagement) relative to comparable pages in the same position tier. A content editor works the queue from the top.

Underneath the ranking, there is a **scoring** step: for each page, we compute a position-adjusted CTR opportunity score. The score combines two things:

1. **How far below its tier's expected CTR** the page sits (the "CTR gap").
2. **How much volume is at stake** (a large gap on a 50-impression page matters less than a small gap on a 10,000-impression page).

The task is *not* plain binary classification ("under-performing yes/no"), because a binary label alone does not rank — it flags ~47% of pages as under-performers (by definition, roughly half sit below the median), which is far too many. The model needs to produce a **continuous score** so we can sort the queue and evaluate whether the top-K pages are genuinely the most actionable.

From the framing-ml-problems skill:

| If your question sounds like… | Task type | Target | Typical metric |
|---|---|---|---|
| "Which ones first?" | Ranking / scoring | a priority score | precision@K |

In [1]:
# Load the starter data and build the working slice for Lane 4.
import pandas as pd
import os

# Handle both Colab (cloned repo) and local paths
local_path = '../../data/raw/content_refresh_anonymized.csv'
colab_path = '/content/ShreeyeshAssignment1/data/raw/content_refresh_anonymized.csv'

if os.path.exists(local_path):
    csv_path = local_path
elif os.path.exists(colab_path):
    csv_path = colab_path
else:
    raise FileNotFoundError('Could not find content_refresh_anonymized.csv')

df = pd.read_csv(csv_path)

# Lane 4 working slice:
# - avg_position > 0  (exclude the 1,205 rows where 0 means "no data")
# - impressions_90d >= 100  (minimum volume so CTR is not pure noise)
lane4 = df[(df['avg_position'] > 0) & (df['impressions_90d'] >= 100)].copy()

print(f'Full dataset: {len(df):,} rows')
print(f'Lane 4 working slice: {len(lane4):,} rows')
print(f'  (excluded {len(df) - len(lane4):,} rows: no position data or < 100 impressions)')
print(f'Distinct clients: {lane4["client_id"].nunique()}')

Full dataset: 30,000 rows
Lane 4 working slice: 22,006 rows
  (excluded 7,994 rows: no position data or < 100 impressions)
Distinct clients: 30


## 2. Target or proxy

**What we predict:** A continuous **CTR opportunity score** — how much a page under-captures clicks relative to peers in the same position tier, weighted by impression volume.

**The proxy label:** Since we do not have a true "this page needs fixing" ground truth (that would require an experiment), we use a **proxy** built from observed CTR behavior:

- For each position tier, compute the **median CTR** among pages with ≥100 impressions.
- A page's **CTR gap** = `tier_median_ctr - page_ctr`. Positive means under-performing.
- The binary proxy label `is_under_ctr` = 1 when a page's CTR falls below its tier median.

**Where the label comes from:** The tier median is computed from **observed CTR** (clicks / impressions, measured by Google Search Console). It is not a product decision flag. However, it is important to acknowledge that this is a **current-window proxy**, not a future outcome — it describes the page's state *now*, not what will happen next. A stronger capstone version could use a future-window label (e.g., "was CTR still below the tier median 30 days later?"), but the proxy is a reasonable starting point.

**Leakage discipline:** `ctr` itself is the quantity we are trying to score, so it is **never a feature**. Neither are `trend_direction` or `trend_pct` (label sources for the decline task). The features we use (impressions, position, word count, content age, engagement, etc.) are all observable *before* anyone decides whether the page needs attention.

In [2]:
# Build the proxy target: is_under_ctr

# Step 1: compute tier-level median CTR
tier_median = lane4.groupby('position_tier')['ctr'].median()
print('Median CTR by position tier (x100 percentages):')
print(tier_median.round(3).to_string())

# Step 2: map tier median to each page and compute gap
lane4['tier_median_ctr'] = lane4['position_tier'].map(tier_median.to_dict())
lane4['ctr_gap'] = lane4['tier_median_ctr'] - lane4['ctr']  # positive = under-performing
lane4['is_under_ctr'] = (lane4['ctr'] < lane4['tier_median_ctr']).astype(int)

print(f'\nProxy label distribution:')
print(lane4['is_under_ctr'].value_counts().rename({0: 'at or above tier median', 1: 'below tier median'}))
print(f'Base rate: {lane4["is_under_ctr"].mean():.1%}')

print(f'\nCTR gap among under-performers:')
under = lane4[lane4['is_under_ctr'] == 1]
print(f'  Median gap: {under["ctr_gap"].median():.3f} percentage points')
print(f'  Q75 gap:    {under["ctr_gap"].quantile(0.75):.3f} pp')
print(f'  Q90 gap:    {under["ctr_gap"].quantile(0.90):.3f} pp')

Median CTR by position tier (x100 percentages):
position_tier
deep        0.00
page_1      0.23
page_3_5    0.06
striking    0.15
top_3       0.19

Proxy label distribution:
is_under_ctr
at or above tier median    11699
below tier median          10307
Name: count, dtype: int64
Base rate: 46.8%

CTR gap among under-performers:
  Median gap: 0.080 percentage points
  Q75 gap:    0.150 pp
  Q90 gap:    0.230 pp


## 3. Success metric

**Primary metric: Precision@50.**

This is the right metric because the output is a **ranked queue** and the real-world constraint is **review capacity**. If a content team can realistically review ~50 pages per cycle, precision@50 measures: *of the top 50 pages the model recommends, how many are genuinely under-capturing clicks?*

**Why precision@K and not accuracy or ROC-AUC:**

- **Accuracy** is misleading here because the base rate is ~47% — a model that always says "under-performing" gets 47% accuracy, which sounds non-trivial but is useless.
- **ROC-AUC** measures the entire ranking curve, but we care most about the **top** of the queue. A model with perfect AUC but mediocre top-50 precision would waste editor time.
- **Precision@50** directly measures the real cost: each false positive in the top 50 is ~30 minutes of wasted editor review.

**Secondary metrics:**

- **Average precision (AP):** Evaluates the full ranking, not just at one cutoff. Useful for comparing models.
- **Precision@20:** For tighter review capacity.
- **A manual top-20 review:** Read the top 20 recommendations and check whether the reason codes make sense and the pages look genuinely actionable.

**What "good" looks like:** The starter pipeline's baseline achieved precision@50 of 0.240 on the decline task (12 of 50 correct), while a random forest reached 0.740 (37 of 50). For this CTR-gap task, a baseline rule should be transparent and beatable. If a trained model achieves precision@50 above 0.70 with client-holdout validation, that would represent a meaningful improvement over a fixed rule.

In [3]:
# Demonstrate what precision@K means concretely.

# A naive rule baseline: rank pages by raw impression volume (descending),
# then check how many of the top 50 are actually under-CTR.
lane4_sorted_by_volume = lane4.sort_values('impressions_90d', ascending=False)
top_50_by_volume = lane4_sorted_by_volume.head(50)
precision_at_50_naive = top_50_by_volume['is_under_ctr'].mean()

print(f'Naive baseline (rank by impressions, take top 50):')
print(f'  Precision@50 = {precision_at_50_naive:.3f}')
print(f'  ({int(precision_at_50_naive * 50)} of 50 are genuinely under-capturing CTR)')

# A slightly smarter rule: rank by ctr_gap * log(impressions)
import numpy as np
lane4_copy = lane4.copy()
lane4_copy['opportunity_score'] = lane4_copy['ctr_gap'] * np.log1p(lane4_copy['impressions_90d'])
lane4_sorted_smart = lane4_copy.sort_values('opportunity_score', ascending=False)
top_50_smart = lane4_sorted_smart.head(50)
precision_at_50_smart = top_50_smart['is_under_ctr'].mean()

print(f'\nSmarter rule (rank by ctr_gap * log(impressions)):')
print(f'  Precision@50 = {precision_at_50_smart:.3f}')
print(f'  ({int(precision_at_50_smart * 50)} of 50 are genuinely under-capturing CTR)')

print(f'\n--> A trained model should beat {precision_at_50_smart:.3f} to earn its complexity.')

Naive baseline (rank by impressions, take top 50):
  Precision@50 = 0.340
  (17 of 50 are genuinely under-capturing CTR)

Smarter rule (rank by ctr_gap * log(impressions)):
  Precision@50 = 1.000
  (50 of 50 are genuinely under-capturing CTR)

--> A trained model should beat 1.000 to earn its complexity.


## 4. The unit of analysis, as a real dataframe

**One row = one content page** (identified by `content_id`) that has measurable search visibility (avg_position > 0, impressions >= 100).

The grain is the same as the starter dataset: one snapshot row per pseudonymized content item, with trailing-90-day aggregated metrics. Each row carries:

- **Identity columns** (for grouping/splitting only): `content_id`, `client_id`
- **Feature candidates** (observable signals): impressions, position, word count, content age, freshness, engagement, scroll, search volume, competition, content type, intent
- **Target proxy** (computed, never a feature): `is_under_ctr`, `ctr_gap`
- **Excluded** (leakage risk): `ctr` (it IS the target), `trend_direction`, `trend_pct`

In [4]:
# Show the actual dataframe: one row = one content page.

display_cols = [
    # Identity
    'content_id', 'client_id',
    # Key features
    'position_tier', 'avg_position', 'impressions_90d', 'clicks_90d',
    'content_type', 'word_count', 'content_age_days', 'days_since_last_update',
    'sessions_90d', 'engagement_rate', 'scroll_rate',
    # Target proxy
    'ctr', 'tier_median_ctr', 'ctr_gap', 'is_under_ctr',
]

# Show 8 rows: mix of under-performers and normal pages
sample = pd.concat([
    lane4[lane4['is_under_ctr'] == 1].sample(4, random_state=42),
    lane4[lane4['is_under_ctr'] == 0].sample(4, random_state=42),
]).sort_values('ctr_gap', ascending=False)

print('Unit of analysis: one row = one content page')
print(f'Working slice: {len(lane4):,} pages, {lane4["client_id"].nunique()} clients')
print()
print(sample[display_cols].to_string(index=False))

Unit of analysis: one row = one content page
Working slice: 22,006 pages, 30 clients

          content_id         client_id position_tier  avg_position  impressions_90d  clicks_90d    content_type  word_count  content_age_days  days_since_last_update  sessions_90d  engagement_rate  scroll_rate  ctr  tier_median_ctr  ctr_gap  is_under_ctr
content_17a540c99ba9 client_f369cb89fc        page_1           9.2              214           0 keyword article      2925.0               147                       8            10             0.00        42.86 0.00             0.23     0.23             1
content_66a9e17988f9 client_f369cb89fc        page_1           3.9              621           1 keyword article      2380.0               117                      20             3             0.00        50.00 0.16             0.23     0.07             1
content_ed7432d0bdff client_8527a891e2      page_3_5          24.6              105           0 keyword article      3777.0               348        

In [5]:
# Verify the grain: one row per content_id (no duplicates).
dupes = lane4.groupby('content_id').size()
n_dupes = (dupes > 1).sum()
print(f'Grain check: {n_dupes} duplicate content_ids found.')
if n_dupes == 0:
    print('Grain is clean: one row = one content page.')
else:
    print('WARNING: duplicates detected -- deduplicate before modeling.')

# Distribution across clients
print(f'\nPages per client (top 5):')
print(lane4['client_id'].value_counts().head(5).to_string())
print(f'\nSmallest 5 clients:')
print(lane4['client_id'].value_counts().tail(5).to_string())
print(f'\n--> Client sizes vary widely: use client_id for grouped train/test splits.')

Grain check: 0 duplicate content_ids found.
Grain is clean: one row = one content page.

Pages per client (top 5):
client_id
client_19581e27de    6579
client_6208ef0f77    3568
client_4e07408562    2278
client_3fdba35f04    2149
client_7f2253d7e2    1019

Smallest 5 clients:
client_id
client_02d20bbd7e    12
client_d59eced1de    10
client_0b918943df     8
client_1a6562590e     2
client_e29c9c180c     1

--> Client sizes vary widely: use client_id for grouped train/test splits.


## 5. Why ML beats a fixed rule here

A fixed rule like *"flag every page with CTR below 0.5% and impressions above 500"* has three problems:

1. **It ignores position.** A page at position 18 naturally gets fewer clicks than one at position 3. Comparing them with the same threshold is unfair. Adjusting for position tier helps, but then the rule just flags ~47% of pages (everyone below the tier median) — still thousands to review.

2. **It cannot rank.** Even after filtering, a rule gives a binary yes/no — it cannot tell you which of the 5,528 flagged pages to review *first*. A model produces a continuous score that orders the queue by expected opportunity.

3. **The pattern involves interacting features.** The data below shows that the CTR gap depends on *combinations* of content type, age, word count, and tier — not on any single threshold. A keyword article at 181–365 days on page 1 has a median gap of 0.13 pp, while a comparison article at the same age has 0.23 pp. Word count shows a non-linear pattern: gap is largest for very short and very long pages, smallest around 2,000–3,000 words. These interactions are exactly the kind of multi-signal, tangled pattern where a tree-based model earns its keep.

In [6]:
# Demonstrate why a fixed rule is not enough.

# Problem 1: Cascading rules still leave thousands of pages
rule1 = lane4[lane4['is_under_ctr'] == 1]
rule2 = rule1[rule1['impressions_90d'] >= 500]
rule3 = rule2[rule2['avg_position'] <= 20]

print('Cascading rule filters:')
print(f'  Below tier median CTR:        {len(rule1):,} pages')
print(f'  + impressions >= 500:          {len(rule2):,} pages')
print(f'  + position <= 20:              {len(rule3):,} pages')
print(f'  --> Still {len(rule3):,} pages. A team reviewing 50/cycle needs')
print(f'      {len(rule3) // 50} cycles. Ranking is essential.')

# Problem 3: Interactions -- gap depends on content_type x age
print('\nInteraction: content_type x age_tier (page_1 under-performers):')
p1_under = lane4[(lane4['position_tier'] == 'page_1') & (lane4['is_under_ctr'] == 1)].copy()
p1_under['ctr_gap'] = p1_under['tier_median_ctr'] - p1_under['ctr']

cross = p1_under.groupby(['content_type', 'age_tier']).agg(
    n=('ctr_gap', 'count'),
    median_gap=('ctr_gap', 'median')
).round(4)
# Show only groups with enough data
cross_filtered = cross[cross['n'] >= 20].reset_index()
print(cross_filtered.to_string(index=False))

# Problem 3 continued: non-linear word count effect
print('\nNon-linear effect: word count vs CTR gap (page_1 under-performers):')
p1_wc = p1_under[p1_under['word_count'].notna()].copy()
p1_wc['wc_bin'] = pd.cut(p1_wc['word_count'], bins=[0, 1000, 2000, 3000, 4000, 10000],
                          labels=['<1k', '1k-2k', '2k-3k', '3k-4k', '4k+'])
wc_gap = p1_wc.groupby('wc_bin', observed=True)['ctr_gap'].agg(['count', 'median']).round(4)
print(wc_gap.to_string())
print()
print('--> The gap varies by content type, age, AND word count simultaneously.')
print('    No single rule threshold captures these interactions.')
print('    A tree-based model can learn them from the data.')

Cascading rule filters:
  Below tier median CTR:        10,307 pages
  + impressions >= 500:          7,159 pages
  + position <= 20:              5,528 pages
  --> Still 5,528 pages. A team reviewing 50/cycle needs
      110 cycles. Ranking is essential.

Interaction: content_type x age_tier (page_1 under-performers):


      content_type age_tier    n  median_gap
comparison article  181-365   35        0.23
comparison article   91-180  146        0.23
    feedly article  181-365   55        0.23
    feedly article   91-180   41        0.23
   keyword article  181-365 1265        0.13
   keyword article    31-90   36        0.23
   keyword article     365+ 1048        0.13
   keyword article   91-180 1634        0.14

Non-linear effect: word count vs CTR gap (page_1 under-performers):
        count  median
wc_bin               
<1k         2   0.230
1k-2k     314   0.225
2k-3k    1316   0.130
3k-4k     692   0.135
4k+       316   0.190

--> The gap varies by content type, age, AND word count simultaneously.
    No single rule threshold captures these interactions.
    A tree-based model can learn them from the data.


## One-paragraph frame

> For a **content editor or SEO specialist** deciding **which pages to review first for metadata or engagement improvements**, we will build a **ranked opportunity queue** from the starter dataset (and later the warehouse release), scoring each page by **how far its CTR falls below the expected CTR for its position tier, weighted by impression volume**. Success is measured by **precision@50** (of the top 50 recommendations, how many are genuinely under-capturing clicks). A wrong recommendation costs **~30 minutes of wasted editor review**; a missed one leaves **compounding click loss on the table**. A plain rule is not enough because the CTR gap depends on **interacting signals** (content type, age, word count, intent, engagement) that no single threshold can capture. We will claim only **observed, directional, decision-support** results — never that a fix *caused* a CTR improvement.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.